# Original cross-attention model with exact-alias token warm-up

This notebook retains the original SapBERT warm-up, NULL-aware token–concept attention, top-\(k\) sparsemax, max pooling, factorized \(\beta=VO\), and outcome training.

The only added stage is an ontology-guided token-classification warm-up:

1. Unique exact ICD-10 names/synonyms are matched in each training note.
2. Tokens inside a matched alias receive that concept as a pseudo-label.
3. Every other valid note token is labeled **NULL**; padding and special tokens are ignored.
4. The attention query/key projections and NULL bias are warmed using token-level cross-entropy.
5. Outcome training then follows the original objective, with a smaller learning rate for attention parameters.

`sample["concepts"]` is never used for training; it is used only for development/test grounding evaluation.

In [ ]:
from __future__ import annotations

import ast
import json
import math
import os
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

from from_n3c import *

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


# -------------------------
# Paths
# -------------------------
CONCEPT_CSV = "../AdaptivePooling_MLHC/df_icd10_with_synonyms.csv"
TRAIN_JSON = "../AdaptivePooling_MLHC/ds_train_chest_trauma_ner.json"
DEV_JSON = "../AdaptivePooling_MLHC/ds_dev_chest_trauma_ner.json"
VAL_JSON = "../AdaptivePooling_MLHC/ds_test_chest_trauma_ner.json"


# -------------------------
# Main hyperparameters
# -------------------------
MODEL_NAME = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
BATCH_SIZE = 4
MAX_LENGTH = 512
DV = 256

BLACKBOX_WARMUP_EPOCHS = 1
BLACKBOX_LR = 1e-5

TOKEN_WARMUP_EPOCHS = 3
TOKEN_WARMUP_LR = 1e-4
NULL_TOKEN_LOSS_WEIGHT = 0.25

OUTCOME_EPOCHS = 2
OUTCOME_LR = 1e-5
ATTENTION_OUTCOME_LR = 1e-6

ATTENTION_TEMPERATURE = 0.07
TOP_K = 4
GATE_MARGIN = 0.85
GATE_TAU = 0.05
NULL_BIAS_INIT = 2.0

LAMBDA_NULL_TARGET = 0.02
NULL_TARGET = 0.95
LAMBDA_ENTROPY = 0.02
LAMBDA_GROUP_LASSO = 1e-3

MAX_ALIASES = 8
MAX_ALIAS_TOKENS = 24


def normalize_code(code: Any) -> str:
    return re.sub(r"[^A-Z0-9]", "", str(code).upper())[:3]


def normalize_alias(text: Any) -> str:
    return re.sub(r"\s+", " ", str(text)).strip(" ,;.")


def parse_synonyms(value: Any) -> List[str]:
    """Parse serialized Python/JSON lists without splitting commas within terms."""
    if value is None:
        return []
    if not isinstance(value, (list, tuple, set, dict)):
        try:
            if pd.isna(value):
                return []
        except (TypeError, ValueError):
            pass

    parsed = value
    for _ in range(2):
        if not isinstance(parsed, str):
            break
        text = parsed.strip()
        if not text:
            return []

        new_value = None
        for candidate in (text, text.strip('"'), text.strip("'")):
            try:
                new_value = ast.literal_eval(candidate)
                break
            except (ValueError, SyntaxError):
                try:
                    new_value = json.loads(candidate)
                    break
                except (ValueError, TypeError, json.JSONDecodeError):
                    continue

        if new_value is None:
            parsed = re.split(r"[|;]", text)
            break
        if new_value == parsed:
            break
        parsed = new_value

    if isinstance(parsed, dict):
        parsed = list(parsed.values())
    elif not isinstance(parsed, (list, tuple, set)):
        parsed = [parsed]

    output = []
    for item in parsed:
        item = normalize_alias(item)
        if item and item.casefold() not in {"nan", "none", "null"}:
            output.append(item)
    return output


def make_aliases(name: str, synonyms: Any) -> List[str]:
    candidates = [
        name,
        *parse_synonyms(synonyms),
        name.replace("(s)", "s"),
        re.sub(r"\[([^]]+)\]", r"\1", name),
    ]

    aliases, seen = [], set()
    for value in candidates:
        value = normalize_alias(value)
        key = value.casefold()
        if value and key not in seen:
            seen.add(key)
            aliases.append(value)
    return aliases


def concept_description(code: str, name: str) -> str:
    """Retain the original manuscript-style concept description."""
    try:
        return (
            f"{name} (Ancestral category: "
            f"{icd10_text(code)}, {infer_chapter_from_code(code)})"
        )
    except Exception:
        return name


def concept_group(code: str) -> str:
    try:
        return str(infer_chapter_from_code(code))
    except Exception:
        return code[:1]


df_concepts = pd.read_csv(CONCEPT_CSV)
required = {"code", "name", "synonyms"}
missing = required - set(df_concepts.columns)
if missing:
    raise ValueError(f"Missing concept columns: {sorted(missing)}")

concepts, seen_codes = [], set()
for _, row in df_concepts.iterrows():
    code = normalize_code(row["code"])
    name = normalize_alias(row["name"])
    if not code or not name or code in seen_codes:
        continue
    seen_codes.add(code)
    concepts.append({
        "id": code,
        "text": concept_description(code, name),
        "name": name,
        "aliases": make_aliases(name, row["synonyms"]),
        "group": concept_group(code),
    })

with open(TRAIN_JSON) as f:
    train_samples = json.load(f)
with open(DEV_JSON) as f:
    dev_samples = json.load(f)
with open(VAL_JSON) as f:
    val_samples = json.load(f)

for samples in (train_samples, dev_samples, val_samples):
    for sample in samples:
        sample["label"] = int(sample["label"] >= 3)

print(f"Concepts: {len(concepts):,}")
print(f"Aliases: {sum(len(c['aliases']) for c in concepts):,}")
print("Train outcome labels:", Counter(s["label"] for s in train_samples))

In [ ]:
class ExactTokenLabeler:
    """
    Build NER-style token pseudo-labels from unique exact ICD aliases.

    Target convention:
      0       = NULL
      1..C    = real concepts 0..C-1 shifted by +1
      -100    = padding/special token; ignored by the loss

    Unmatched valid note tokens are explicitly assigned NULL.
    Overlapping aliases are resolved longest-first.
    """

    def __init__(
        self,
        tokenizer,
        concepts,
        max_aliases: int = 8,
        max_alias_tokens: int = 24,
    ):
        self.tokenizer = tokenizer
        self.num_concepts = len(concepts)
        self.max_alias_tokens = int(max_alias_tokens)
        self.special_ids = set(tokenizer.all_special_ids)
        self.cache: Dict[Tuple[int, ...], List[Tuple[int, int, int]]] = {}

        pattern_to_concepts: Dict[Tuple[int, ...], set] = defaultdict(set)
        pattern_to_alias: Dict[Tuple[int, ...], str] = {}

        for concept_idx, concept in enumerate(concepts):
            for alias in concept["aliases"][:max_aliases]:
                alias = normalize_alias(alias)
                if not alias:
                    continue

                token_ids = tuple(
                    tokenizer(alias, add_special_tokens=False)["input_ids"]
                )
                if not token_ids or len(token_ids) > self.max_alias_tokens:
                    continue

                # Avoid highly unstable very-short one-token aliases, while
                # retaining ordinary clinical terms and uppercase abbreviations.
                surface_terms = re.findall(r"[A-Za-z0-9]+", alias)
                compact = re.sub(r"[^A-Za-z0-9]", "", alias)
                is_abbreviation = (
                    compact.isupper()
                    and 2 <= len(compact) <= 8
                    and compact.casefold() not in {"nos", "nec"}
                )
                if (
                    len(token_ids) == 1
                    and len(compact) < 5
                    and not is_abbreviation
                ):
                    continue

                pattern_to_concepts[token_ids].add(concept_idx)
                pattern_to_alias[token_ids] = alias

        # Only token sequences mapping to exactly one concept are safe targets.
        self.by_first_token: Dict[int, List[Tuple[Tuple[int, ...], int]]] = defaultdict(list)
        covered_concepts = set()

        for pattern, concept_indices in pattern_to_concepts.items():
            if len(concept_indices) != 1:
                continue
            concept_idx = next(iter(concept_indices))
            self.by_first_token[pattern[0]].append((pattern, concept_idx))
            covered_concepts.add(concept_idx)

        # Longest match wins at every start position.
        for first_token in self.by_first_token:
            self.by_first_token[first_token].sort(
                key=lambda item: len(item[0]),
                reverse=True,
            )

        self.n_unique_patterns = sum(
            len(items) for items in self.by_first_token.values()
        )
        self.n_covered_concepts = len(covered_concepts)

    def _match_one(
        self,
        token_ids: Tuple[int, ...],
    ) -> List[Tuple[int, int, int]]:
        if token_ids in self.cache:
            return self.cache[token_ids]

        candidates = []
        n = len(token_ids)

        for start, first_token in enumerate(token_ids):
            for pattern, concept_idx in self.by_first_token.get(first_token, []):
                end = start + len(pattern)
                if end <= n and token_ids[start:end] == pattern:
                    candidates.append((start, end, concept_idx))

        # Resolve overlap globally by longest span, then earliest location.
        candidates.sort(key=lambda x: (-(x[1] - x[0]), x[0], x[2]))
        occupied = [False] * n
        selected = []

        for start, end, concept_idx in candidates:
            if any(occupied[start:end]):
                continue
            selected.append((start, end, concept_idx))
            for position in range(start, end):
                occupied[position] = True

        selected.sort(key=lambda x: x[0])
        self.cache[token_ids] = selected
        return selected

    @torch.no_grad()
    def __call__(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        input_ids_cpu = input_ids.detach().cpu()
        attention_cpu = attention_mask.detach().cpu().bool()

        targets = torch.full(
            input_ids_cpu.shape,
            fill_value=-100,
            dtype=torch.long,
        )

        for row_idx in range(input_ids_cpu.shape[0]):
            valid_length = int(attention_cpu[row_idx].sum())
            ids = tuple(input_ids_cpu[row_idx, :valid_length].tolist())

            # All valid non-special tokens start as NULL.
            for position, token_id in enumerate(ids):
                if token_id not in self.special_ids:
                    targets[row_idx, position] = 0

            for start, end, concept_idx in self._match_one(ids):
                for position in range(start, end):
                    if ids[position] not in self.special_ids:
                        targets[row_idx, position] = concept_idx + 1

        return targets

    def summary(self) -> Dict[str, int]:
        return {
            "unique_alias_patterns": self.n_unique_patterns,
            "concepts_with_unique_alias": self.n_covered_concepts,
            "cache_size": len(self.cache),
        }


class TextDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


def annotation_code(annotation) -> str:
    if isinstance(annotation, dict):
        raw = annotation.get("code", annotation.get("id"))
    elif isinstance(annotation, (list, tuple)) and len(annotation) >= 2:
        raw = annotation[1]
    else:
        raw = None
    return normalize_code(raw) if raw is not None else ""


def make_loader(
    samples,
    tokenizer,
    batch_size=4,
    max_length=512,
    shuffle=False,
    token_labeler: Optional[ExactTokenLabeler] = None,
    include_concepts=False,
):
    concept_to_idx = {concept["id"]: i for i, concept in enumerate(concepts)}

    def collate(batch):
        encoded = tokenizer(
            [sample["txt"] for sample in batch],
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        encoded["labels"] = torch.tensor(
            [sample["label"] for sample in batch],
            dtype=torch.long,
        )

        if token_labeler is not None:
            encoded["token_targets"] = token_labeler(
                encoded["input_ids"],
                encoded["attention_mask"],
            )

        if include_concepts:  # held-out evaluation only
            concept_labels = torch.zeros(
                len(batch),
                len(concepts),
                dtype=torch.bool,
            )
            for row_idx, sample in enumerate(batch):
                for annotation in sample.get("concepts", []):
                    concept_idx = concept_to_idx.get(annotation_code(annotation))
                    if concept_idx is not None:
                        concept_labels[row_idx, concept_idx] = True
            encoded["concept_labels"] = concept_labels

        return encoded

    return DataLoader(
        TextDataset(samples),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate,
    )


@torch.no_grad()
def summarize_token_targets(loader, max_batches=None):
    n_positive = n_null = n_ignored = 0

    for batch_idx, batch in enumerate(loader):
        targets = batch["token_targets"]
        n_positive += int((targets > 0).sum())
        n_null += int((targets == 0).sum())
        n_ignored += int((targets == -100).sum())

        if max_batches is not None and batch_idx + 1 >= max_batches:
            break

    return {
        "matched_concept_tokens": n_positive,
        "null_tokens": n_null,
        "ignored_tokens": n_ignored,
        "matched_fraction_of_valid": (
            n_positive / max(n_positive + n_null, 1)
        ),
    }

In [ ]:
def sparsemax(logits: torch.Tensor, dim: int = -1) -> torch.Tensor:
    if dim < 0:
        dim += logits.dim()

    z = logits.transpose(dim, -1)
    original_shape = z.shape
    z = z.reshape(-1, original_shape[-1])

    z_sorted, _ = torch.sort(z, descending=True, dim=-1)
    z_cumsum = z_sorted.cumsum(dim=-1)
    k = torch.arange(
        1,
        z.shape[-1] + 1,
        device=z.device,
        dtype=z.dtype,
    ).view(1, -1)

    support = 1 + k * z_sorted > z_cumsum
    k_z = support.sum(dim=-1, keepdim=True).clamp(min=1)
    tau = (
        z_cumsum.gather(-1, (k_z - 1).long()) - 1
    ) / k_z.to(z.dtype)

    return torch.clamp(z - tau, min=0.0).reshape(
        original_shape
    ).transpose(dim, -1)


@torch.no_grad()
def build_concept_embeddings(
    concept_texts,
    tokenizer,
    encoder,
    device,
    batch_size=4,
    max_length=256,
):
    encoder.eval()
    rows = []

    for start in range(0, len(concept_texts), batch_size):
        tokens = tokenizer(
            concept_texts[start:start + batch_size],
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)
        rows.append(
            encoder(**tokens).last_hidden_state[:, 0].detach().cpu()
        )

    return torch.cat(rows, dim=0)


@dataclass
class AVOOutput:
    logits: torch.Tensor
    token_logits: torch.Tensor       # unpruned pre-softmax logits, (B,L,C+1)
    A: torch.Tensor                  # original sparse attention, (B,L,C+1)
    V: torch.Tensor
    O: torch.Tensor
    sim: torch.Tensor
    A_pool: torch.Tensor
    AV_pool: torch.Tensor


class MentionAlignedAVOHead(nn.Module):
    """Original NULL-aware attention head plus trainable Q/K projections."""

    def __init__(
        self,
        concept_emb,
        dv=256,
        num_outputs=2,
        temperature=0.07,
        gate_margin=0.85,
        gate_tau=0.05,
        top_k=4,
        attn_activation="sparsemax",
        null_bias_init=2.0,
    ):
        super().__init__()
        C, H = concept_emb.shape
        self.C, self.H, self.dv = C, H, dv
        self.temperature = float(temperature)
        self.gate_margin = float(gate_margin)
        self.gate_tau = float(gate_tau)
        self.top_k = top_k
        self.attn_activation = attn_activation

        self.register_buffer(
            "concept_emb",
            concept_emb.detach().clone(),
        )

        # Identity initialization preserves the original cosine matcher at start.
        self.Wq = nn.Linear(H, H, bias=False)
        self.Wk = nn.Linear(H, H, bias=False)
        nn.init.eye_(self.Wq.weight)
        nn.init.eye_(self.Wk.weight)

        self.Wv = nn.Linear(H, dv, bias=False)
        self.O = nn.Parameter(torch.randn(dv, num_outputs) * 0.02)
        self.bias = nn.Parameter(torch.zeros(num_outputs))
        self.null_bias = nn.Parameter(
            torch.tensor(float(null_bias_init))
        )

    def _attention(self, logits):
        if self.attn_activation == "softmax":
            return torch.softmax(logits, dim=-1)
        if self.attn_activation == "sparsemax":
            return sparsemax(logits, dim=-1)
        raise ValueError("attn_activation must be softmax or sparsemax")

    def forward(self, token_embs, token_mask):
        q = F.normalize(self.Wq(token_embs), dim=-1)
        k = F.normalize(self.Wk(self.concept_emb), dim=-1)
        sim = torch.einsum("blh,ch->blc", q, k)

        logits_real_full = sim / max(self.temperature, 1e-6)
        max_similarity = sim.max(dim=-1, keepdim=True).values
        null_logit = (
            (self.gate_margin - max_similarity)
            / max(self.gate_tau, 1e-6)
            + self.null_bias
        )

        # These unpruned logits receive the NER-style token loss.
        token_logits = torch.cat(
            [null_logit, logits_real_full],
            dim=-1,
        )

        # The original prediction pathway retains top-k sparse attention.
        logits_real_attention = logits_real_full
        if self.top_k is not None and self.top_k < self.C:
            _, top_indices = torch.topk(
                logits_real_attention,
                k=self.top_k,
                dim=-1,
            )
            keep = torch.zeros_like(
                logits_real_attention,
                dtype=torch.bool,
            )
            keep.scatter_(-1, top_indices, True)
            logits_real_attention = logits_real_attention.masked_fill(
                ~keep,
                -1e9,
            )

        attention_logits = torch.cat(
            [null_logit, logits_real_attention],
            dim=-1,
        )
        attention_logits = attention_logits.masked_fill(
            ~token_mask.unsqueeze(-1),
            -1e9,
        )
        attention_logits[..., 0] = attention_logits[..., 0].masked_fill(
            ~token_mask,
            0.0,
        )

        A = self._attention(attention_logits)
        A = A.masked_fill(~token_mask.unsqueeze(-1), 0.0)
        A_pool = A.max(dim=1).values

        V_real = self.Wv(self.concept_emb)
        V = torch.cat(
            [V_real.new_zeros(1, self.dv), V_real],
            dim=0,
        )
        AV_pool = A_pool @ V
        outcome_logits = AV_pool @ self.O + self.bias

        return AVOOutput(
            logits=outcome_logits,
            token_logits=token_logits,
            A=A,
            V=V,
            O=self.O,
            sim=sim,
            A_pool=A_pool,
            AV_pool=AV_pool,
        )


class MentionAlignedAVOModel(nn.Module):
    def __init__(self, encoder, head, tokenizer):
        super().__init__()
        self.text_encoder = encoder
        self.head = head
        self.special_ids = tokenizer.all_special_ids

        for parameter in self.text_encoder.parameters():
            parameter.requires_grad = False

    def train(self, mode=True):
        super().train(mode)
        self.text_encoder.eval()
        return self

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None,
    ):
        kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        token_embs = self.text_encoder(**kwargs).last_hidden_state
        token_mask = attention_mask.bool()
        for token_id in self.special_ids:
            token_mask &= input_ids != token_id

        return self.head(token_embs, token_mask), token_mask


def balanced_token_grounding_loss(
    token_logits,
    token_targets,
    null_weight=0.25,
):
    """
    Average matched-concept and NULL losses separately so the large number of
    unmatched tokens does not collapse the model to NULL.
    """
    flat_logits = token_logits.reshape(-1, token_logits.shape[-1])
    flat_targets = token_targets.reshape(-1)

    per_token = F.cross_entropy(
        flat_logits,
        flat_targets,
        ignore_index=-100,
        reduction="none",
    )

    positive_mask = flat_targets > 0
    null_mask = flat_targets == 0

    positive_loss = (
        per_token[positive_mask].mean()
        if positive_mask.any()
        else per_token.new_zeros(())
    )
    null_loss = (
        per_token[null_mask].mean()
        if null_mask.any()
        else per_token.new_zeros(())
    )

    return positive_loss + null_weight * null_loss


def mention_regularization(
    out,
    token_mask,
    lambda_null=0.02,
    null_target=0.95,
    lambda_entropy=0.02,
    eps=1e-8,
):
    A_null = out.A[..., 0]
    A_real = out.A[..., 1:]

    mean_null = A_null[token_mask].mean()
    null_loss = lambda_null * (
        mean_null - null_target
    ).pow(2)

    nonnull = (1.0 - A_null).clamp_min(eps).unsqueeze(-1)
    real_distribution = (A_real / nonnull).clamp_min(eps)
    entropy = -(
        real_distribution * real_distribution.log()
    ).sum(dim=-1)

    weights = (1.0 - A_null).detach()
    denominator = (
        weights[token_mask].sum().clamp_min(1.0)
    )
    entropy_loss = lambda_entropy * (
        entropy[token_mask] * weights[token_mask]
    ).sum() / denominator

    return null_loss + entropy_loss


def build_groups(concepts):
    by_group = defaultdict(list)
    for concept_idx, concept in enumerate(concepts):
        by_group[concept["group"]].append(concept_idx)
    return list(by_group.values())


def group_lasso(beta_with_null, groups, lambda_=1e-3):
    penalty = beta_with_null.new_zeros(())
    for group in groups:
        if not group:
            continue
        indices = torch.tensor(
            group,
            device=beta_with_null.device,
            dtype=torch.long,
        ) + 1
        penalty += (
            math.sqrt(len(group))
            * beta_with_null.index_select(0, indices).norm()
        )
    return lambda_ * penalty


@torch.no_grad()
def evaluate_outcome(model, loader, device):
    model.eval()
    labels, probabilities = [], []

    for batch in loader:
        tensors = {
            key: value.to(device)
            for key, value in batch.items()
            if isinstance(value, torch.Tensor)
        }
        out, _ = model(
            tensors["input_ids"],
            tensors["attention_mask"],
            tensors.get("token_type_ids"),
        )
        labels.append(tensors["labels"].cpu().numpy())
        probabilities.append(
            torch.softmax(out.logits, dim=-1)[:, 1].cpu().numpy()
        )

    y = np.concatenate(labels)
    p = np.concatenate(probabilities)
    return {
        "AUROC": roc_auc_score(y, p),
        "AUPR": average_precision_score(y, p),
    }


@torch.no_grad()
def evaluate_grounding(
    model,
    loader,
    device,
    ks=(1, 5, 10),
    contribution=False,
):
    model.eval()
    hits = {k: 0 for k in ks}
    n_valid = 0

    for batch in loader:
        tensors = {
            key: value.to(device)
            for key, value in batch.items()
            if isinstance(value, torch.Tensor)
        }
        out, _ = model(
            tensors["input_ids"],
            tensors["attention_mask"],
            tensors.get("token_type_ids"),
        )

        scores = out.A_pool[:, 1:]
        if contribution:
            beta = (out.V @ out.O)[1:]
            beta_contrast = beta[:, 1] - beta[:, 0]
            scores = scores * beta_contrast.unsqueeze(0)

        labels = tensors["concept_labels"].bool()
        valid = labels.any(dim=1)
        if not valid.any():
            continue

        top = scores[valid].topk(
            min(max(ks), scores.shape[1]),
            dim=1,
        ).indices
        labels = labels[valid]
        n_valid += int(valid.sum())

        for k in ks:
            kk = min(k, top.shape[1])
            hits[k] += int(
                labels.gather(1, top[:, :kk]).any(dim=1).sum()
            )

    prefix = "contributor" if contribution else "presence"
    return {
        "n_labeled_notes": n_valid,
        **{
            f"{prefix}_hit@{k}": hits[k] / max(n_valid, 1)
            for k in ks
        },
    }


@torch.no_grad()
def evaluate_token_pseudo_labels(model, loader, device):
    model.eval()
    correct_positive = n_positive = 0
    correct_null = n_null = 0

    for batch in loader:
        tensors = {
            key: value.to(device)
            for key, value in batch.items()
            if isinstance(value, torch.Tensor)
        }
        out, _ = model(
            tensors["input_ids"],
            tensors["attention_mask"],
            tensors.get("token_type_ids"),
        )

        prediction = out.token_logits.argmax(dim=-1)
        targets = tensors["token_targets"]

        positive = targets > 0
        null = targets == 0

        correct_positive += int(
            (prediction[positive] == targets[positive]).sum()
        )
        n_positive += int(positive.sum())

        correct_null += int((prediction[null] == 0).sum())
        n_null += int(null.sum())

    return {
        "matched_token_accuracy": (
            correct_positive / max(n_positive, 1)
        ),
        "null_token_accuracy": correct_null / max(n_null, 1),
        "n_matched_tokens": n_positive,
        "n_null_tokens": n_null,
    }

In [ ]:
# Tokenizer, exact token labeler, and loaders
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
token_labeler = ExactTokenLabeler(
    tokenizer=tokenizer,
    concepts=concepts,
    max_aliases=MAX_ALIASES,
    max_alias_tokens=MAX_ALIAS_TOKENS,
)
print("Exact token labeler:", token_labeler.summary())

train_loader = make_loader(
    train_samples,
    tokenizer,
    BATCH_SIZE,
    MAX_LENGTH,
    shuffle=True,
)
dev_loader = make_loader(
    dev_samples,
    tokenizer,
    BATCH_SIZE,
    MAX_LENGTH,
)
val_loader = make_loader(
    val_samples,
    tokenizer,
    BATCH_SIZE,
    MAX_LENGTH,
)

train_token_loader = make_loader(
    train_samples,
    tokenizer,
    BATCH_SIZE,
    MAX_LENGTH,
    shuffle=True,
    token_labeler=token_labeler,
)
dev_token_loader = make_loader(
    dev_samples,
    tokenizer,
    BATCH_SIZE,
    MAX_LENGTH,
    token_labeler=token_labeler,
)

dev_grounding_loader = make_loader(
    dev_samples,
    tokenizer,
    BATCH_SIZE,
    MAX_LENGTH,
    include_concepts=True,
)
val_grounding_loader = make_loader(
    val_samples,
    tokenizer,
    BATCH_SIZE,
    MAX_LENGTH,
    include_concepts=True,
)

print("Training pseudo-targets:", summarize_token_targets(train_token_loader))


# Original task-adaptive outcome warm-up
class BlackBoxLM(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(encoder.config.hidden_size, 2)

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None,
    ):
        kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        hidden = self.encoder(**kwargs).last_hidden_state
        return self.head(hidden[:, 0])


encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
blackbox = BlackBoxLM(encoder).to(DEVICE)
blackbox_optimizer = torch.optim.AdamW(
    blackbox.parameters(),
    lr=BLACKBOX_LR,
)

for epoch in range(1, BLACKBOX_WARMUP_EPOCHS + 1):
    blackbox.train()
    total_loss = 0.0

    for batch in train_loader:
        tensors = {
            key: value.to(DEVICE)
            for key, value in batch.items()
            if isinstance(value, torch.Tensor)
        }
        logits = blackbox(
            tensors["input_ids"],
            tensors["attention_mask"],
            tensors.get("token_type_ids"),
        )
        loss = F.cross_entropy(logits, tensors["labels"])

        blackbox_optimizer.zero_grad(set_to_none=True)
        loss.backward()
        blackbox_optimizer.step()
        total_loss += loss.item()

    print(
        f"Blackbox warm-up {epoch}: "
        f"loss={total_loss / max(len(train_loader), 1):.4f}"
    )

In [ ]:
# Build original concept-attention model after the outcome warm-up
encoder = blackbox.encoder
concept_embeddings = build_concept_embeddings(
    [concept["text"] for concept in concepts],
    tokenizer,
    encoder,
    DEVICE,
    batch_size=4,
    max_length=256,
).to(DEVICE)

head = MentionAlignedAVOHead(
    concept_emb=concept_embeddings,
    dv=DV,
    num_outputs=2,
    temperature=ATTENTION_TEMPERATURE,
    gate_margin=GATE_MARGIN,
    gate_tau=GATE_TAU,
    top_k=TOP_K,
    attn_activation="sparsemax",
    null_bias_init=NULL_BIAS_INIT,
).to(DEVICE)

model = MentionAlignedAVOModel(
    encoder=encoder,
    head=head,
    tokenizer=tokenizer,
).to(DEVICE)

groups = build_groups(concepts)
del concept_embeddings
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "Before token warm-up:",
    evaluate_token_pseudo_labels(
        model,
        dev_token_loader,
        DEVICE,
    ),
)
print(
    "Dev presence before token warm-up:",
    evaluate_grounding(
        model,
        dev_grounding_loader,
        DEVICE,
    ),
)

In [ ]:
# ============================================================
# Stage 1: exact-match NER-style token warm-up
# ============================================================

# Only attention-identification parameters are updated.
for parameter in model.parameters():
    parameter.requires_grad = False
for parameter in model.head.Wq.parameters():
    parameter.requires_grad = True
for parameter in model.head.Wk.parameters():
    parameter.requires_grad = True
model.head.null_bias.requires_grad_(True)

token_optimizer = torch.optim.AdamW(
    [
        *model.head.Wq.parameters(),
        *model.head.Wk.parameters(),
        model.head.null_bias,
    ],
    lr=TOKEN_WARMUP_LR,
)

for epoch in range(1, TOKEN_WARMUP_EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for batch in train_token_loader:
        tensors = {
            key: value.to(DEVICE)
            for key, value in batch.items()
            if isinstance(value, torch.Tensor)
        }

        out, _ = model(
            tensors["input_ids"],
            tensors["attention_mask"],
            tensors.get("token_type_ids"),
        )

        token_loss = balanced_token_grounding_loss(
            out.token_logits,
            tensors["token_targets"],
            null_weight=NULL_TOKEN_LOSS_WEIGHT,
        )

        token_optimizer.zero_grad(set_to_none=True)
        token_loss.backward()
        token_optimizer.step()
        total_loss += token_loss.item()

    print(
        f"Token warm-up {epoch}/{TOKEN_WARMUP_EPOCHS}: "
        f"loss={total_loss / max(len(train_token_loader), 1):.4f}"
    )
    print(
        "  exact-token diagnostic:",
        evaluate_token_pseudo_labels(
            model,
            dev_token_loader,
            DEVICE,
        ),
    )
    print(
        "  dev presence grounding:",
        evaluate_grounding(
            model,
            dev_grounding_loader,
            DEVICE,
        ),
    )

# Save the warmed attention state for drift reporting.
attention_reference = {
    "Wq": model.head.Wq.weight.detach().clone(),
    "Wk": model.head.Wk.weight.detach().clone(),
    "null_bias": model.head.null_bias.detach().clone(),
}

In [ ]:
# ============================================================
# Stage 2: original outcome training with a smaller attention LR
# ============================================================

for parameter in model.parameters():
    parameter.requires_grad = False

# Original outcome layer
for parameter in model.head.Wv.parameters():
    parameter.requires_grad = True
model.head.O.requires_grad_(True)
model.head.bias.requires_grad_(True)

# Attention parameters remain adaptable, but at a smaller learning rate.
for parameter in model.head.Wq.parameters():
    parameter.requires_grad = True
for parameter in model.head.Wk.parameters():
    parameter.requires_grad = True
model.head.null_bias.requires_grad_(True)

outcome_parameters = [
    *model.head.Wv.parameters(),
    model.head.O,
    model.head.bias,
]
attention_parameters = [
    *model.head.Wq.parameters(),
    *model.head.Wk.parameters(),
    model.head.null_bias,
]

optimizer = torch.optim.AdamW([
    {
        "params": outcome_parameters,
        "lr": OUTCOME_LR,
    },
    {
        "params": attention_parameters,
        "lr": ATTENTION_OUTCOME_LR,
    },
])


def maximum_attention_drift():
    return max(
        float(
            (model.head.Wq.weight - attention_reference["Wq"])
            .abs()
            .max()
        ),
        float(
            (model.head.Wk.weight - attention_reference["Wk"])
            .abs()
            .max()
        ),
        float(
            (model.head.null_bias - attention_reference["null_bias"])
            .abs()
            .max()
        ),
    )


for epoch in range(1, OUTCOME_EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        tensors = {
            key: value.to(DEVICE)
            for key, value in batch.items()
            if isinstance(value, torch.Tensor)
        }

        out, token_mask = model(
            tensors["input_ids"],
            tensors["attention_mask"],
            tensors.get("token_type_ids"),
        )

        outcome_loss = F.cross_entropy(
            out.logits,
            tensors["labels"],
        )
        regularization_loss = mention_regularization(
            out,
            token_mask,
            lambda_null=LAMBDA_NULL_TARGET,
            null_target=NULL_TARGET,
            lambda_entropy=LAMBDA_ENTROPY,
        )
        beta = out.V @ out.O
        sparsity_loss = group_lasso(
            beta,
            groups,
            lambda_=LAMBDA_GROUP_LASSO,
        )

        loss = (
            outcome_loss
            + regularization_loss
            + sparsity_loss
        )

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(
        f"Outcome epoch {epoch}/{OUTCOME_EPOCHS}: "
        f"loss={total_loss / max(len(train_loader), 1):.4f}, "
        f"maximum_attention_drift={maximum_attention_drift():.2e}"
    )
    print(
        "  outcome:",
        evaluate_outcome(model, dev_loader, DEVICE),
    )
    print(
        "  exact-token diagnostic:",
        evaluate_token_pseudo_labels(
            model,
            dev_token_loader,
            DEVICE,
        ),
    )
    print(
        "  presence:",
        evaluate_grounding(
            model,
            dev_grounding_loader,
            DEVICE,
        ),
    )
    print(
        "  contributor:",
        evaluate_grounding(
            model,
            dev_grounding_loader,
            DEVICE,
            contribution=True,
        ),
    )


print(
    "Final validation/test outcome:",
    evaluate_outcome(model, val_loader, DEVICE),
)
print(
    "Final validation/test presence grounding:",
    evaluate_grounding(
        model,
        val_grounding_loader,
        DEVICE,
    ),
)
print(
    "Final validation/test contributor grounding:",
    evaluate_grounding(
        model,
        val_grounding_loader,
        DEVICE,
        contribution=True,
    ),
)